The goal of this script is to concat the lissajous data, but some things have to be kept in mind for a succesfull concatenation.
1. a standard file system should allow the script to easily specify the variables of the measurements, those variables are:

    1.1. reaction type
        - co2-splitting
        - drm
    1.2. supplier
        - e.g.: ugent, uhasselt, vito, etc.
    1.3. material
        - e.g.: SiO2@Titania, Al203@Fe2O3, etc.
    1.4. pwr-const/sei-const
    1.5. residence time - measurement number - plasma state
        - xy.zs-ab-state

**Aggregation level — read before changing the combine cell.**
The Lissajous data is nested: **AC cycle → snapshot (`.psdata` capture) → reactor
run (`measurement_number`)**. Cycles within a capture, and repeat captures within a
run, are *pseudo-replicates* (not independent experiments). The electrical combine
therefore collapses to **one row per reactor run**: cycles → snapshot mean, then
snapshots → run mean. Per metric `X` it emits `X` (run mean) and `X_sd` (the
**between-snapshot** standard deviation = the run's measurement error). `X_sd` is a
*fitting weight only* — do **not** draw it as an error bar (pseudo-replication).
Downstream fitting (`electrical_lin_fit.ipynb`) consumes this run-level file.

In [ ]:
import os
import pandas as pd

In [ ]:
# Combine electrical (Lissajous) results at the REACTOR-RUN level.
#
# Replication hierarchy (least -> most independent):
#   AC cycle  ->  snapshot (.psdata capture)  ->  reactor run (measurement_number)
# Cycles within a capture, and repeat captures within a run, are pseudo-replicates.
# We therefore collapse in two stages:
#   1. cycles    -> snapshot : mean of numeric metrics within each .psdata capture
#   2. snapshots -> run      : mean (the run value) + std (between-snapshot SD)
# The between-snapshot SD (suffix `_sd`) is the run's MEASUREMENT error: keep it as a
# fitting weight, NOT as a visualisation error bar (it is a pseudo-replicate spread).

# Determine the parent folder
parent_folder = r'C:\Users\Sande\OneDrive - Universiteit Antwerpen\plasmacatdesign\co2-splitting\uhasselt'

# Identity columns that define one independent reactor run.
run_keys = [
    'project_name', 'reaction_type', 'material_supplier', 'material_name',
    'wattage_const', 'residence_time_s', 'measurement_number', 'plasma_state',
]

# ---- Stage 1: cycles -> snapshot (one row per .psdata capture) -----------------
snapshot_frames = []
for subdir, _, files in os.walk(parent_folder):
    if 'lissajous_calculations_no_outliers.parquet' in files:
        df = pd.read_parquet(
            os.path.join(subdir, 'lissajous_calculations_no_outliers.parquet')
        )
        metric_cols = [
            c for c in df.columns
            if c not in run_keys + ['psdata_file_name', 'measurement_date', 'cycle']
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        snap = (
            df.groupby(['psdata_file_name'] + run_keys, dropna=False)[metric_cols]
              .mean()
              .reset_index()
        )
        snapshot_frames.append(snap)

snapshots = (
    pd.concat(snapshot_frames, ignore_index=True) if snapshot_frames else None
)

# ---- Stage 2: snapshots -> reactor run (one row per measurement_number) --------
metric_cols = [
    c for c in snapshots.columns
    if c not in run_keys + ['psdata_file_name']
    and pd.api.types.is_numeric_dtype(snapshots[c])
]
grp = snapshots.groupby(run_keys, dropna=False)
run_mean = grp[metric_cols].mean()
run_sd = grp[metric_cols].std(ddof=1).add_suffix('_sd')      # between-snapshot SD
n_snapshots = grp.size().rename('n_snapshots')
combined_data = pd.concat([run_mean, run_sd, n_snapshots], axis=1).reset_index()

# Save the run-level combined data
combined_data.to_parquet(
    os.path.join(parent_folder, 'uhasselt_co2_spl_electrical_data_combined.parquet'),
    index=False,
    compression='gzip',
)

print(
    f"Electrical data combined at reactor-run level: {combined_data.shape[0]} runs, "
    f"{len(metric_cols)} metrics (mean + _sd), median n_snapshots/run = "
    f"{int(combined_data['n_snapshots'].median())}."
)

In [ ]:
# Initialize an empty list to store all the data
data_frames = []

# Loop through the files in the parent folder and its subdirectories
for subdir, _, files in os.walk(parent_folder):
    for filename in files:
        if filename == 'profile_data.parquet':
            file_path = os.path.join(subdir, filename)
            # Read the CSV file and append the DataFrame to the list
            df = pd.read_parquet(
                file_path
            )
            data_frames.append(df)

# Concatenate all data frames into a single DataFrame
combined_data = pd.concat(
    data_frames, ignore_index=True
) if data_frames else None

# Save the combined data to a new CSV file
combined_data.to_parquet(
    os.path.join(
        parent_folder,
        'uhasselt_co2_spl_profile_data_combined.parquet'
    ),
    index=False,
    compression='gzip'
)

print(
    "All profile files have been successfully combined!"
)

In [ ]:
# Clear variables, specific for Jupyter notebooks
%reset -f